In [ ]:
!virtualenv aor_submission

In [ ]:
!aor_submission\Scripts\activate

In [ ]:
!pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from src.utils import *
from src.data_processing import *
from src.chains_data_processing import *
from src.feature_creation import *

In [ ]:
match_ids = [
    1886347,
    1899585,
    1925299,
    1953632,
    1996435,    
    2006229,
    2011166,
    2013725,
    2015213,
    2017461,
]

pitch_width = 105
pitch_height = 68

skillcorner_opendata_media_url = "https://media.githubusercontent.com/media/SkillCorner/opendata/master/data/matches/"
skillcorner_opendata_raw_url = "https://raw.githubusercontent.com/SkillCorner/opendata/master/data/matches/"

skillcorner = SkillCornerData()
fc = FeatureCreation()
cdp = GKChains()
utils = Utils()

In [ ]:
td_df = skillcorner.load_tracking_data(skillcorner_opendata_media_url, match_ids=match_ids)

pm_list = skillcorner.prepare_metadata_files(skillcorner_opendata_raw_url, match_ids=match_ids)
pm_df = skillcorner.load_metadata(pm_list)
pm_df = skillcorner.process_players_metadata_dataframe(pm_df)

de_list = skillcorner.prepare_dynamic_events_files(skillcorner_opendata_raw_url, match_ids=match_ids)
de_df = skillcorner.load_dynamic_events_data(de_list)
de_df = skillcorner.filter_player_possessions(de_df)

In [ ]:
# Step 7: Analyze all possession chains
gk_chains = []
i = 0

for (match_id, chain_id), chain_group in de_df.groupby(['match_id', 'possession_chain_id']):
    chain_result = cdp.analyze_gk_chain(chain_group, i)
    i += 1
    if chain_result:
        gk_chains.append(chain_result)

# Convert to DataFrame
gk_chains_df = pd.DataFrame(gk_chains)

In [ ]:
cdp.get_gk_chains_info(gk_chains_df=gk_chains_df)

In [ ]:
synced_df = gk_chains_df.merge(
    td_df,
    left_on=["match_id", "frame_end"],
    right_on=["match_id", "frame"],
    suffixes=("_event", "_tracking"),
)

In [ ]:
synced_df.head()

In [ ]:
synced_df = utils.apply_coords_to_df(synced_df, pitch_width, pitch_height)

In [ ]:
model_features_df = fc.create_features_dataframe(synced_df.possession_index.unique(), synced_df)

In [ ]:
test_model1 = smf.glm(formula="gk_led_to_final_third ~ d2 + max_x_reached ",
                      data = model_features_df,
                      family = sm.families.Binomial()).fit()

summary_text = test_model1.summary()
print(summary_text)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(model_features_df[test_model1.params.keys()[1:]], model_features_df['gk_led_to_final_third'], test_size=0.33, random_state=42)

# Train the RandomForest classifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

# Predict probabilities for the test set
y_probs = clf.predict_proba(model_features_df[test_model1.params.keys()[1:]])[::, 1]

model_features_df["GKLaunch"] = y_probs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

roc_auc = roc_auc_score(model_features_df['gk_led_to_final_third'], y_probs)
print(f'ROC-AUC Score: {roc_auc:.2f}')

fpr, tpr, thresholds = roc_curve(model_features_df['gk_led_to_final_third'], y_probs)
axes[0].plot(fpr, tpr)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate') 

axes[1].axis("off")
axes[1].text(
    0, 1, summary_text,
    fontsize=10,
    va="top",
    family="monospace"
)

plt.tight_layout()
plt.show()

In [ ]:
result_df = model_features_df.groupby(["gkId"])["GKLaunch"].sum().sort_values(ascending=False).reset_index()
result_df = result_df.merge(
    pm_df,
    how="left",
    left_on=["gkId"],
    right_on=["id"]
)

In [ ]:
fc.add_normalized_metrics_to_dataframe(result_df)

In [ ]:
result_df.sort_values('GKLaunchPer90', ascending=False)[['short_name', 'team_name', 'total_minutes_played', 'total_minutes_played_tip', 'total_minutes_played_otip', 'GKLaunchPer90', 'GKLaunchPer30TIP', 'GKLaunchPer30OTIP']]